In [10]:
# KV cache means During autoregressive generation, save the Key and Value vectors from previous tokens 
# so we do not recompute them again and again. It is used during inference/generation, not usually during normal full-sequence training.

# Q = what am I looking for?
# K = what information does each token offer for matching?
# V = what information do I retrieve if I attend to that token?

# input_ids =[I, love, deep, learning]

# During training, we pass the whole sequence at once:

# The model computes Q/K/V for all tokens at once. No KV cache is needed because we are not repeatedly decoding one token at a time
# Q/K/V for I
# Q/K/V for love
# Q/K/V for deep
# Q/K/V for learning

# During generation, we decode one token at a time
# Suppose the prompt is: I love
# The model predicts the next token: deep
# Then the new input becomes: I love deep
# Then the model predicts: learning

# At each step, we recompute everything.

# Without KV cache:

# Step 1:
# input: I love
# compute K/V for: I, love
# predict: deep

# Step 2:
# input: I love deep
# compute K/V again for: I, love, deep
# predict: learning

# Step 3:
# input: I love deep learning
# compute K/V again for: I, love, deep, learning
# predict next token

# This wastes computation because I and love do not change. Their K/V vectors are the same every time.

# With KV cache we save the old K and V:

# Step 1:
# input: I love
# compute K/V for: I, love
# save K/V in cache
# predict: deep

# Cache contains:
# K_cache = [K_I, K_love]
# V_cache = [V_I, V_love]

# Step 2 
# input: deep 
# compute K/V only for: deep
# append to cache
# predict: learning

# Cache contains:
# K_cache = [K_I, K_love, K_deep]
#  V_cache = [V_I, V_love, V_deep]

# input: learning only
# compute K/V only for: learning
# append to cache
# predict next token

# Why do we cache K and V, but not Q?

#For each new token, we need a new query:

# Q_new = what is the current token looking for?

# But the new token must attend to all previous tokens, so it needs old K and V:

# K_cache = keys for all previous tokens
# V_cache = values for all previous tokens


In [11]:
from dataclasses import dataclass
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 64
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [12]:
class PositionalEmbeddingLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )

        self.position_embeddings = nn.Embedding(
            num_embeddings=config.max_seq_len,
            embedding_dim=config.d_model,
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids, position_ids=None):
        """
        input_ids:
            [B, T]

        position_ids:
            [B, T] or None

        returns:
            x: [B, T, D]
        """

        B, T = input_ids.shape

        if position_ids is None:
            position_ids = torch.arange(
                T,
                device=input_ids.device,
            ).unsqueeze(0).expand(B, T)
            # [B, T]

        token_embs = self.token_embeddings(input_ids)
        # [B, T, D]

        pos_embs = self.position_embeddings(position_ids)
        # [B, T, D]

        x = token_embs + pos_embs

        return self.dropout(x)

In [13]:
class CausalSelfAttentionWithKVCache(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.d_head = config.d_head
        self.max_seq_len = config.max_seq_len

        self.q_proj = nn.Linear(config.d_model, config.d_model)
        self.k_proj = nn.Linear(config.d_model, config.d_model)
        self.v_proj = nn.Linear(config.d_model, config.d_model)

        self.o_proj = nn.Linear(config.d_model, config.d_model)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    
    def make_causal_mask(self, T, S, past_len, device):
        """
        T:
            query length

        S:
            total key length = past_len + T

        past_len:
            number of cached previous tokens

        returns:
            causal_mask: [1, 1, T, S]
        """

        query_positions = torch.arange(
            past_len,
            past_len + T,
            device=device,
        )
        # [T]

        key_positions = torch.arange(
            S,
            device=device,
        )
        # [S]

        causal_mask = key_positions[None, :] <= query_positions[:, None]
        # [T, S]

        causal_mask = causal_mask[None, None, :, :]
        # [1, 1, T, S]

        return causal_mask
        

    def forward(
        self,
        x,
        attention_mask=None,
        layer_past=None,
        use_cache=False,
    ):
        """
        x:
            [B, T, D]

        attention_mask:
            [B, S], where S = total key length
            1 = valid token, 0 = padding

        layer_past:
            None or (past_k, past_v)

            past_k: [B, H, S_old, d_head]
            past_v: [B, H, S_old, d_head]

        use_cache:
            if True, return present K/V cache
        """

        B, T, D = x.shape

        assert D == self.d_model

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # [B, T, D]

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        # [B, H, T, d_head]

        if layer_past is not None:
            past_k, past_v = layer_past
            # [B, H, S_old, d_head]

            past_len = past_k.shape[2]

            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)
            # [B, H, S_old + T, d_head]

        else:
            past_len = 0
            k_all = k
            v_all = v

        S = k_all.shape[2]

        scores = q @ k_all.transpose(-2, -1)
        # [B, H, T, S]

        scores = scores / math.sqrt(self.d_head)

        causal_mask = self.make_causal_mask(
            T=T,
            S=S,
            past_len=past_len,
            device=x.device,
        )
        # [1, 1, T, S]

        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].bool()
            # [B, 1, 1, S]

            combined_mask = causal_mask & key_padding_mask
            # [B, 1, T, S]
        else:
            combined_mask = causal_mask

        scores = scores.masked_fill(
            ~combined_mask,
            torch.finfo(scores.dtype).min,
        )

        attn_weights = torch.softmax(scores, dim=-1)
        # [B, H, T, S]

        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v_all
        # [B, H, T, d_head]

        out = out.transpose(1, 2).contiguous()
        # [B, T, H, d_head]

        out = out.view(B, T, D)
        # [B, T, D]

        out = self.o_proj(out)
        out = self.resid_dropout(out)

        present = None

        if use_cache:
            present = (k_all, v_all)

        if use_cache:
            return out, present

        return out

In [14]:
class FeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

In [15]:
class DecoderBlockWithKVCache(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttentionWithKVCache(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(
        self,
        x,
        attention_mask=None,
        layer_past=None,
        use_cache=False,
    ):
        """
        x:
            [B, T, D]

        layer_past:
            None or (past_k, past_v)
        """

        if use_cache:
            attn_out, present = self.attn(
                self.ln_1(x),
                attention_mask=attention_mask,
                layer_past=layer_past,
                use_cache=True,
            )
        else:
            attn_out = self.attn(
                self.ln_1(x),
                attention_mask=attention_mask,
                layer_past=None,
                use_cache=False,
            )
            present = None

        x = x + attn_out

        x = x + self.mlp(self.ln_2(x))

        if attention_mask is not None:
            # attention_mask may be [B, S_total].
            # x only has query length T, so use the last T positions.
            query_mask = attention_mask[:, -x.shape[1]:]
            x = x * query_mask[:, :, None].to(x.dtype)

        if use_cache:
            return x, present

        return x

In [16]:
class GPTDecoderWithKVCache(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = PositionalEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            DecoderBlockWithKVCache(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # Weight tying
        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        past_key_values=None,
        use_cache=False,
    ):
        """
        Training:
            input_ids: [B, T]
            attention_mask: [B, T]
            labels: [B, T]
            past_key_values=None
            use_cache=False

        Cached inference:
            input_ids: [B, T_new]
            past_key_values: list of layer caches
            use_cache=True
        """

        B, T = input_ids.shape

        if past_key_values is None:
            past_key_values = [None] * len(self.blocks)
            past_len = 0
        else:
            past_len = past_key_values[0][0].shape[2]
            # past_key_values[0][0] is past_k from layer 0:
            # [B, H, S_old, d_head]

        position_ids = torch.arange(
            past_len,
            past_len + T,
            device=input_ids.device,
        ).unsqueeze(0).expand(B, T)
        # [B, T]

        x = self.embedding(
            input_ids=input_ids,
            position_ids=position_ids,
        )
        # [B, T, D]

        present_key_values = [] if use_cache else None

        for layer_idx, block in enumerate(self.blocks):
            layer_past = past_key_values[layer_idx]

            if use_cache:
                x, present = block(
                    x,
                    attention_mask=attention_mask,
                    layer_past=layer_past,
                    use_cache=True,
                )

                present_key_values.append(present)

            else:
                x = block(
                    x,
                    attention_mask=attention_mask,
                    layer_past=None,
                    use_cache=False,
                )

        x = self.final_ln(x)

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "logits": logits,
            "loss": loss,
            "past_key_values": present_key_values,
        }

    def sample_next_token(
        self,
        next_token_logits,
        temperature=1.0,
        top_k=None,
        top_p=None,
    ):
        if temperature == 0:
            return torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

        next_token_logits = next_token_logits / temperature

        if top_k is not None:
            top_k = min(top_k, next_token_logits.size(-1))

            values, _ = torch.topk(
                next_token_logits,
                k=top_k,
                dim=-1,
            )

            min_values = values[:, -1].unsqueeze(-1)

            next_token_logits = torch.where(
                next_token_logits < min_values,
                torch.full_like(next_token_logits, float("-inf")),
                next_token_logits,
            )

        if top_p is not None:
            assert 0.0 < top_p <= 1.0

            sorted_logits, sorted_indices = torch.sort(
                next_token_logits,
                descending=True,
                dim=-1,
            )

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p

            sorted_indices_to_remove[:, 1:] = (
                sorted_indices_to_remove[:, :-1].clone()
            )
            sorted_indices_to_remove[:, 0] = False

            indices_to_remove = torch.zeros_like(
                next_token_logits,
                dtype=torch.bool,
            )

            indices_to_remove.scatter_(
                dim=-1,
                index=sorted_indices,
                src=sorted_indices_to_remove,
            )

            next_token_logits = next_token_logits.masked_fill(
                indices_to_remove,
                float("-inf"),
            )

        probs = torch.softmax(next_token_logits, dim=-1)

        return torch.multinomial(
            probs,
            num_samples=1,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens,
        temperature=1.0,
        top_k=None,
        top_p=None,
        eos_token_id=None,
    ):
        """
        Cached autoregressive generation.

        For simplicity, pass an unpadded prompt.
        """

        self.eval()

        generated = input_ids

        past_key_values = None

        # First iteration uses the full prompt.
        # Later iterations use only the latest generated token.
        current_input_ids = input_ids

        for _ in range(max_new_tokens):
            outputs = self(
                input_ids=current_input_ids,
                attention_mask=None,
                labels=None,
                past_key_values=past_key_values,
                use_cache=True,
            )

            logits = outputs["logits"]
            past_key_values = outputs["past_key_values"]

            next_token_logits = logits[:, -1, :]

            next_token = self.sample_next_token(
                next_token_logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1,
            )

            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break

            # After prefill, only feed the new token.
            current_input_ids = next_token

        return generated

In [17]:
torch.manual_seed(42)

batch_size = 3
seq_len = 8
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, seq_len),
)

attention_mask = (
    torch.arange(seq_len).unsqueeze(0)
    < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(
    attention_mask == 0,
    -100,
)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=32,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,
)

model = GPTDecoderWithKVCache(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

logits = outputs["logits"]
loss = outputs["loss"]

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss)

loss.backward()

print("\nGradient exists for q_proj?")
print(model.blocks[0].attn.q_proj.weight.grad is not None)

print("\nGradient exists for k_proj?")
print(model.blocks[0].attn.k_proj.weight.grad is not None)

print("\nGradient exists for v_proj?")
print(model.blocks[0].attn.v_proj.weight.grad is not None)

lengths:
tensor([7, 4, 5])

input_ids:
tensor([[59, 91, 66, 26, 78, 86,  3,  0],
        [77, 81, 82, 91,  0,  0,  0,  0],
        [32, 77,  9, 65, 51,  0,  0,  0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0]])

labels:
tensor([[  59,   91,   66,   26,   78,   86,    3, -100],
        [  77,   81,   82,   91, -100, -100, -100, -100],
        [  32,   77,    9,   65,   51, -100, -100, -100]])

logits shape:
torch.Size([3, 8, 100])

loss:
tensor(22.6713, grad_fn=<NllLossBackward0>)

Gradient exists for q_proj?
True

Gradient exists for k_proj?
True

Gradient exists for v_proj?
True


In [18]:
prompt_len = int(attention_mask[0].sum().item())
prompt = input_ids[0:1, :prompt_len]

print("\nprompt:")
print(prompt)

generated = model.generate(
    input_ids=prompt,
    max_new_tokens=5,
    temperature=1.0,
    top_k=10,
    top_p=0.9,
)

print("\ngenerated:")
print(generated)

print("\ngenerated shape:")
print(generated.shape)


prompt:
tensor([[59, 91, 66, 26, 78, 86,  3]])

generated:
tensor([[59, 91, 66, 26, 78, 86,  3,  3,  3,  3,  3,  3]])

generated shape:
torch.Size([1, 12])


In [19]:
model.eval()

with torch.no_grad():
    prompt_len = int(attention_mask[0].sum().item())
    prompt = input_ids[0:1, :prompt_len]

    test_next_token = torch.tensor(
        [[7]],
        dtype=torch.long,
        device=prompt.device,
    )

    full_input = torch.cat(
        [prompt, test_next_token],
        dim=1,
    )

    # ----------------------------------
    # Full forward
    # ----------------------------------

    full_outputs = model(
        input_ids=full_input,
        attention_mask=None,
        labels=None,
        past_key_values=None,
        use_cache=False,
    )

    full_last_logits = full_outputs["logits"][:, -1, :]

    # ----------------------------------
    # Cached forward
    # ----------------------------------

    prefill_outputs = model(
        input_ids=prompt,
        attention_mask=None,
        labels=None,
        past_key_values=None,
        use_cache=True,
    )

    past_key_values = prefill_outputs["past_key_values"]

    decode_outputs = model(
        input_ids=test_next_token,
        attention_mask=None,
        labels=None,
        past_key_values=past_key_values,
        use_cache=True,
    )

    cached_last_logits = decode_outputs["logits"][:, -1, :]

    max_diff = (full_last_logits - cached_last_logits).abs().max()

print("\nMax difference between full forward and cached forward:")
print(max_diff) # Expected value should be very small, usually around: 1e-6 to 1e-5


Max difference between full forward and cached forward:
tensor(1.9073e-06)
